<a href="https://colab.research.google.com/github/ankitta-singh/machinelearning/blob/main/14_TPOT_Hyperparameter_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install tpot

In [2]:
import tpot

print(tpot.__version__)

1.1.0


In [3]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tpot import TPOTRegressor

In [4]:
from google.colab import files

uploaded = files.upload()

Saving house-prices-advanced-regression-techniques.zip to house-prices-advanced-regression-techniques.zip


In [8]:
import zipfile
with zipfile.ZipFile("house-prices-advanced-regression-techniques.zip", "r") as zip_ref:
  zip_ref.extractall("house data")

In [9]:
df = pd.read_csv("house data/train.csv")
df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [10]:
X = df.drop("SalePrice", axis=1)
y = df["SalePrice"]

In [11]:
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1460, 80)
y shape: (1460,)


In [12]:
print("Total missing values:", X.isnull().sum().sum())

Total missing values: 7829


In [13]:
X.isnull().sum().sort_values(ascending=False).head(10)

,0
PoolQC,1453
MiscFeature,1406
Alley,1369
Fence,1179
MasVnrType,872
FireplaceQu,690
LotFrontage,259
GarageType,81
GarageQual,81
GarageFinish,81


In [14]:
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = X.select_dtypes(include=["object"]).columns
print("Numerical columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))

Numerical columns: 37
Categorical columns: 43


In [15]:
print("Total columns:", len(numeric_cols) + len(categorical_cols))

Total columns: 80


In [16]:
X[numeric_cols] = X[numeric_cols].fillna(
    X[numeric_cols].median()
)

In [17]:
X[categorical_cols] = X[categorical_cols].fillna("Missing")

In [18]:
print("Remaining missing values:", X.isnull().sum().sum())

Remaining missing values: 0


In [19]:
X = pd.get_dummies(X, drop_first=True)

In [20]:
print("X shape:", X.shape)

X shape: (1460, 261)


In [21]:
print("Missing values:", X.isnull().sum().sum())

Missing values: 0


In [22]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [23]:
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (1168, 261)
X_test : (292, 261)
y_train: (1168,)
y_test : (292,)


In [24]:
import inspect
from tpot import TPOTRegressor

print(inspect.signature(TPOTRegressor))

(search_space='linear', scorers=['neg_mean_squared_error'], scorers_weights=[1], cv=10, other_objective_functions=[], other_objective_functions_weights=[], objective_function_names=None, bigger_is_better=True, categorical_features=None, memory=None, preprocessing=False, max_time_mins=60, max_eval_time_mins=10, n_jobs=1, validation_strategy='none', validation_fraction=0.2, early_stop=None, warm_start=False, periodic_checkpoint_folder=None, verbose=2, memory_limit=None, client=None, random_state=None, allow_inner_regressors=None, **tpotestimator_kwargs)


In [25]:
import tpot.config

print(dir(tpot.config))

['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'autoqtl_builtins', 'classifiers', 'classifiers_sklearnex', 'get_configspace', 'get_search_space', 'imputers', 'mdr_configs', 'regressors', 'regressors_sklearnex', 'selectors', 'special_configs', 'template_search_spaces', 'transformers']


In [26]:
from tpot.config import get_search_space

print(get_search_space)

<function get_search_space at 0x784b0fbaa7a0>


In [27]:
import inspect

print(inspect.signature(get_search_space))

(name, n_classes=3, n_samples=1000, n_features=100, random_state=None, return_choice_pipeline=True, base_node=<class 'tpot.search_spaces.nodes.estimator_node.EstimatorNode'>, n_jobs=1)


In [28]:
from tpot.config import template_search_spaces

print(template_search_spaces)


<module 'tpot.config.template_search_spaces' from '/usr/local/lib/python3.12/dist-packages/tpot/config/template_search_spaces.py'>


In [30]:
import tpot.config.template_search_spaces as ts

print(dir(ts))

['BaseEstimator', 'Callable', 'ChoicePipeline', 'ChoicePipelineIndividual', 'ConfigurationSpace', 'DynamicLinearPipeline', 'DynamicLinearPipelineIndividual', 'DynamicUnionPipeline', 'DynamicUnionPipelineIndividual', 'EstimatorNode', 'EstimatorNodeIndividual', 'FSSIndividual', 'FSSNode', 'FeatureSetSelector', 'Generator', 'GeneticFeatureSelectorIndividual', 'GeneticFeatureSelectorNode', 'GraphKey', 'GraphPipelineIndividual', 'GraphSearchPipeline', 'List', 'MaskSelector', 'SearchSpace', 'SelectorMixin', 'SequentialPipeline', 'SequentialPipelineIndividual', 'SklearnIndividual', 'TreePipeline', 'TreePipelineIndividual', 'Tuple', 'TupleIndex', 'Union', 'UnionPipeline', 'UnionPipelineIndividual', 'WrapperPipeline', 'WrapperPipelineIndividual', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'choice', 'config', 'copy', 'default_hyperparameter_parser', 'dynamic_linear', 'dynamicunion', 'estimator_node', 'final', 'fss_node', 'genetic_fea

In [31]:
from tpot.config.template_search_spaces import get_linear_search_space

import inspect

print(inspect.signature(get_linear_search_space))

(classification=True, inner_predictors=True, cross_val_predict_cv=0, **get_search_space_params)


In [32]:
from tpot.config.template_search_spaces import get_linear_search_space

search_space = get_linear_search_space(
    classification=False,
    inner_predictors=False
)

In [33]:
print(type(search_space))

<class 'tpot.search_spaces.pipelines.sequential.SequentialPipeline'>


In [34]:
tpot = TPOTRegressor(
    search_space=search_space,
    scorers=["neg_mean_squared_error"],
    cv=5,
    max_time_mins=10,
    max_eval_time_mins=2,
    n_jobs=1,
    verbose=2,
    random_state=42
)

In [35]:
tpot.fit(X_train, y_train)

INFO:distributed.http.proxy:To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
INFO:distributed.scheduler:State start
INFO:distributed.scheduler:  Scheduler at:     tcp://127.0.0.1:45475
INFO:distributed.scheduler:  dashboard at:  http://127.0.0.1:8787/status
INFO:distributed.scheduler:Registering Worker plugin shuffle
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:40861'
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:46629 name: 0
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:46629
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:58502
INFO:distributed.scheduler:Receive client connection: Client-54a6c45e-9a71-11f1-8c8d-0242ac1c000c
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:58512
Generation: : 0it [00:00, ?it/s]INFO:distributed.scheduler:Client Client-54a6c45e-9a71-11f1-8c8d-0242ac1c000c requests t

TPOTRegressor(cv=5, max_eval_time_mins=2, max_time_mins=10, random_state=42,
              search_space=<tpot.search_spaces.pipelines.sequential.SequentialPipeline object at 0x784b0f199580>)

In [37]:
print("Best TPOT Pipeline:")
print(tpot.fitted_pipeline_)

Best TPOT Pipeline:
Pipeline(steps=[('standardscaler', StandardScaler()),
                ('selectfwe', SelectFwe(alpha=0.0139504436408)),
                ('featureunion',
                 FeatureUnion(transformer_list=[('featureunion',
                                                 FeatureUnion(transformer_list=[('quantiletransformer',
                                                                                 QuantileTransformer(n_quantiles=145,
                                                                                                     output_distribution=np.str_('uniform'))),
                                                                                ('zerocount',
                                                                                 ZeroCount())])),
                                                ('passthrough',
                                                 Passthrough())])),
                ('lgbmregressor',
                 LGBMRegressor(boosting_

In [38]:
y_pred = tpot.predict(X_test)
print("Predictions generated successfully!")
print("Number of predictions:", len(y_pred))

Predictions generated successfully!
Number of predictions: 292


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [39]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("===== TPOT TEST RESULTS =====")
print("MAE :", mae)
print("MSE :", mse)
print("RMSE:", rmse)
print("R²  :", r2)
print("R² %:", r2 * 100)

===== TPOT TEST RESULTS =====
MAE : 18069.14332937996
MSE : 1000184108.7091336
RMSE: 31625.687482000034
R²  : 0.8696034455781089
R² %: 86.96034455781088


I used TPOTRegressor for  hyperparameter optimization. I provided a regression search space and used 5-fold cross-validation with negative MSE as the optimization metric. TPOT automatically searched different pipelines and hyperparameter configurations and selected an optimized LightGBM-based pipeline.

---






